# Project 2 - Sales Analysis - Machine Learning

## Objectives

- Transform the Date column to pandas datetime
- Read data from combined csv file created from ETL processes
- Create machine learning model to predict new hypothesis data
- Show plots to attempt to answer new hypothesis


## Inputs

CSV files used:

Sales_Combined_DataSet_Visualization.csv


## Outputs

Pipeline for use in streamlit 
Plots to show the data and attempt to answer the hypothesis


## Additional Comments

- Developed an experimental ETL library which is in this project (modETL_library.py)   
  Has lots of cool features so will be interesting to see how it works "in the field"

Used various AI tools to help with the visualisation process:
- ChatGPT
- GitHub Copilot

See Documents/What_AI_Used_For.md for more details.

Needed to install:

pip install nbformat

For plotly visualisations to work


- Isolated the visualisations into a seperate notebook to keep the file size as low as possible.


## Initalise Working Environment

In [14]:
%matplotlib inline

#iimport libraries
import os
import streamlit as st
import pandas as pd
import seaborn as sns
import plotly.express as px
import pathlib
import scipy.stats as stats
import matplotlib.pyplot as plt
import nbformat
from matplotlib.ticker import MultipleLocator

#for ML experiments
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_curve,
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)

import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np
#from pyexpat import features

#for creating the pipeline file needed by streamlit/Heroku
import joblib

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT


# Hypothesis Being Tested

- What are the predicted sales for store types by month for next year?
- What are the predicted sales for stores in areas of high unemployment by month for next year?

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [15]:
#model and image paths
CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_PATH = modGlobal.CNST_STR_PIPELINESPATH + "/linear_regression_hypothesis12_test_pipeline.pkl"
CNST_STR_LINEAR_PLOT_HYPOTHESIS12_TEST_PATH = modGlobal.CNST_STR_MLREPORTIMAGESPATH + "/linear_regression_hypothesis12_test_forecast.png"
CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_PATH = modGlobal.CNST_STR_PIPELINESPATH + "/forest_regression_hypothesis12_test_pipeline.pkl"
CNST_STR_FOREST_PLOT_HYPOTHESIS12_TEST_PATH = modGlobal.CNST_STR_MLREPORTIMAGESPATH + "/forest_regression_hypothesis12_test_forecast.png"

CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_PATH = modGlobal.CNST_STR_PIPELINESPATH + "/linear_regression_hypothesis12_pipeline.pkl"
CNST_STR_LINEAR_PLOT_HYPOTHESIS12_PATH = modGlobal.CNST_STR_MLREPORTIMAGESPATH + "/linear_regression_hypothesis12_forecast.png"
CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_PATH = modGlobal.CNST_STR_PIPELINESPATH + "/forest_regression_hypothesis12_pipeline.pkl"
CNST_STR_FOREST_PLOT_HYPOTHESIS12_PATH = modGlobal.CNST_STR_MLREPORTIMAGESPATH + "/forest_regression_hypothesis12_forecast.png"

#for copying into streamlit folder
CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH =  "streamlit/assets/pipelines/linear_regression_hypothesis12_test_pipeline.pkl"
CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH = "streamlit/assets/pipelines/randomforest_hypothesis12_test_pipeline.pkl"

CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH =  "streamlit/assets/pipelines/linear_regression_hypothesis12_pipeline.pkl"
CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH =  "streamlit/assets/pipelines/randomforest_hypothesis12_pipeline.pkl"

#DataFrame vars for visualisation
dfSalesDataML = None
dfSalesDataML_Work = None
dfTempML = None

#ML vars
dfTrain = pd.DataFrame()
dfTest = pd.DataFrame()
dfXTrain = pd.DataFrame()
dfyTrain = pd.DataFrame()
dfXTest = pd.DataFrame()
dfyTest = pd.DataFrame()
dfPlot = pd.DataFrame()
dfPredictedSales = pd.DataFrame()
dfPrevious = pd.DataFrame()
dfPreviousSales = pd.DataFrame()
dfTestSales = pd.DataFrame()
dfActualSales = pd.DataFrame()
dfFeatures = pd.DataFrame()
dfSales = pd.DataFrame()
dfStores = pd.DataFrame()
dfFeaturesStores = pd.DataFrame()
dfForecast = pd.DataFrame()
dfXForecast = pd.DataFrame()
dfForecastSales = pd.DataFrame()
dfStoreDepts = pd.DataFrame()

lstNumericFeatures = list()
lstCategoricalFeatures = list()
lstFeatures = list()
lstStores = list()
lstSales = list()
lstMarkdown = list()

strTarget = ""

objModel = None
objPipeline = None
objMissingPreviousSales = None
objNumericTransformer = None
objCategoricalTransformer = None
objPreProcessor = None
objyPred = None
objyPred = None
objMissingColumns = None

fltMAE = 0.0
fltRMSE = 0.0
fltR2 = 0.0

intTestYear = 0
intPreviousYear = 0

#other vars
intYear = 0
fig = None
ax = None
intNum = 0
dictDataFrames = dict()
objPipeline = None
objModel = None
X = None
X_Const = None
y = None


## Set Current Directory To Base Project Directory

In [16]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project2-SalesAnalysis


# Section 1 - Read Data From CSV File

- Read csv file

## Read csv File Into Variable For Processing and Copy Into dfSales_Combined_DataSet_Work

In [17]:
#read file from working files folder
#ETL library returns a dictionary of all files in the folder with the attribute name
#set to the actual csv filename
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
#open into DataFrame which can be re-used during visualisations for filtering etc
dfSalesDataML = dictDataFrames["Sales_Combined_DataSet_Visualisation.csv"]

4 csv Files Read Into DataFrames

DataFrames Created:
Sales_Combined_DataSet_Visualisation.csv
Features_DataSet_Visualisation.csv
Sales_DataSet_Visualisation.csv
Stores_DataSet_Visualisation.csv




## Transform Date Column To Datetime Format

In [18]:
#transform Date to datetime
dfSalesDataML["Date"] = pd.to_datetime(dfSalesDataML["Date"], format="%d/%m/%Y")

#check schema
dfSalesDataML.dtypes

#create a working copy of the base DataFrame
dfSalesDataML_Work = dfSalesDataML.copy()

# Initailise Work DataFrame

In [19]:
#setup work DataFrame
dfSalesDataML_Work = dfSalesDataML.copy() 


# Hypothesis 12

What are the predicted sales for store types by month for next year?


# Linear Regression Model - Hypothesis 12 - Test

First run a test comparing predicitons for last years values with the actual values to see how accurate the model is

In [ ]:
#linear regression test compare last years sales with predicted values for same year
#saves pipeline into main pipelines folder and the same in the streamlit folder

dfSalesDataML_Work = dfSalesDataML.copy() 

# Remove rows where the date could not be converted
dfSalesDataML_Work = dfSalesDataML_Work.dropna(subset=["Date"])

#sprinkle some feature engineering
dfSalesDataML_Work["Year"] = dfSalesDataML_Work["Date"].dt.year
dfSalesDataML_Work["Month"] = dfSalesDataML_Work["Date"].dt.month
dfSalesDataML_Work["Day"] = dfSalesDataML_Work["Date"].dt.day
dfSalesDataML_Work["DayOfWeek"] = dfSalesDataML_Work["Date"].dt.dayofweek
dfSalesDataML_Work["WeekOfYear"] = dfSalesDataML_Work["Date"].dt.isocalendar().week.astype(int)
dfSalesDataML_Work["Quarter"] = dfSalesDataML_Work["Date"].dt.quarter

#cyclical month features <- thanks to StackOverflow user "chimpsarehungry" for the solution
dfSalesDataML_Work["MonthSin"] = np.sin(2 * np.pi * dfSalesDataML_Work["Month"] / 12)
dfSalesDataML_Work["MonthCos"] = np.cos(2 * np.pi * dfSalesDataML_Work["Month"] / 12)

# Convert boolean to integer
dfSalesDataML_Work["IsHoliday"] = dfSalesDataML_Work["IsHoliday"].astype(int)


#create test and traing data
#
#IMPORTANT:
#2012 is the test year
#2013 is NOT used as the test data

dfTrain = dfSalesDataML_Work[dfSalesDataML_Work["Year"] < 2012].copy()
dfTest = dfSalesDataML_Work[dfSalesDataML_Work["Year"] == 2012].copy()

#show what is being used
print(f"Training Rows: {len(dfTrain)}")
print(f"Testing Rows: {len(dfTest)}")
print()
#show test and training date ranges
print(f"Training Date Range: {dfTrain['Date'].min()} To {dfTrain['Date'].max()}")
print(f"Testing Date Range: {dfTest['Date'].min()} To {dfTest['Date'].max()}")
print()

#defaine features to use
lstFeatures = [
    "Store",
    "Dept",
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

#define target
strTarget = "Weekly_Sales"

#setup X and y for training and testing
dfXtrain = dfTrain[lstFeatures]
dfyTrain = dfTrain[strTarget]

dfXTest = dfTest[lstFeatures]
dfyTest = dfTest[strTarget]


#pre-processing configuration
lstNumericFeatures = [
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

lstCategoricalFeatures = [
    "Store",
    "Dept"
]

#no reason not to use the median
objNumericTransformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

#like most frequent to get better balanced results
objCategoricalTransformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

objPreProcessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            objNumericTransformer,
            lstNumericFeatures
        ),
        (
            "categorical",
            objCategoricalTransformer,
            lstCategoricalFeatures
        )
    ]
)

#create linear regression model
objModel = LinearRegression()

#configure pipeline
objPipeline = Pipeline(
    steps=[
        ("preprocessor", objPreProcessor),
        ("model", objModel)
    ]
)


#train the model
print("Training Linear Regression...")
objPipeline.fit(dfXtrain, dfyTrain)
print()

#run prediciton
objyPred = objPipeline.predict(dfXTest)

#evaluate performance of the model
fltMAE = mean_absolute_error(dfyTest, objyPred)

fltRMSE = np.sqrt(
    mean_squared_error(dfyTest, objyPred)
)

fltR2 = r2_score(dfyTest, objyPred)

#show results
print("Linear Regression Results")
print("=" * 30)

print(f"Mean Absolute Error (MAE):  {fltMAE:,.2f}")
print(f"Root Mean Squared Error (RMSE): {fltRMSE:,.2f}")
print(f"R-Squared (R²):   {fltR2:.4f}")
print()

#show OLS model summary for Weekly_Sales
ols_model = smf.ols("Weekly_Sales ~ Store + Dept + IsHoliday + Year + Month + Day + DayOfWeek + WeekOfYear + Quarter + MonthSin + MonthCos", 
                    data=dfSalesDataML_Work).fit()
print(ols_model.summary())
print()


#save pipelie to main folder and streamlit folder
#save to main folder
joblib.dump(
    objPipeline,
    CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_PATH
)

#save to streamlit folder
joblib.dump(
    objPipeline,
    CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH
)

#tell user saved
print("Pipeline Saved To:")
print(CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_PATH)
print(CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH)


Training Rows: 294132
Testing Rows: 127438

Training Date Range: 2010-02-05 00:00:00 To 2011-12-30 00:00:00
Testing Date Range: 2012-01-06 00:00:00 To 2012-10-26 00:00:00

Training Linear Regression...

Linear Regression Results
Mean Absolute Error (MAE):  7,981.97
Root Mean Squared Error (RMSE): 12,215.22
R-Squared (R²):   0.6951

                            OLS Regression Results                            
Dep. Variable:           Weekly_Sales   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     1386.
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:40:58   Log-Likelihood:            -4.8200e+06
No. Observations:              421570   AIC:                         9.640e+06
Df Residuals:                  421559   BIC:                         9.640e+06
Df Model:                         

# Observations - Linear Regression Model - Hypothesis 12 - Test

The R Squared value is 0.6951 which is far enough from 0 to suggest the model is a good fit for the data  
For science will compare the results with random forest

The Mean Absolute Error (MAE) is 7,981.97 which means on average the model predicted sales differed from the actual sales by  
approximately 7,981.97 per store and department, considering the size of average stores sales not too bad a variance  

Root Mean Squared Error (RMSE) is 12.215.22 the fact it is higher than MAE suggests that while most predications are  
close to the actual values, there are some occasion where the model is less accurate   

These are usually caused by outliers

The OLS regression summary shows that a model based soley on Weekly_Sales shows no significant correlation with the other   
variables in the model, which is to be expected as the model is based on Weekly_Sales, but it shown here for completeness



# Linear Regression Plot - Hypothesis 12 - Test

Visualise the model results

In [ ]:
#now plot the results

dfSalesDataML_Work = dfSalesDataML.copy() 
dfSalesDataML_Work = dfSalesDataML_Work.dropna(subset=["Date"])

#sprinkle some feature engineering
dfSalesDataML_Work["Year"] = dfSalesDataML_Work["Date"].dt.year
dfSalesDataML_Work["Month"] = dfSalesDataML_Work["Date"].dt.month
dfSalesDataML_Work["Day"] = dfSalesDataML_Work["Date"].dt.day
dfSalesDataML_Work["DayOfWeek"] = dfSalesDataML_Work    ["Date"].dt.dayofweek
dfSalesDataML_Work["WeekOfYear"] = dfSalesDataML_Work["Date"].dt.isocalendar().week.astype(int)
dfSalesDataML_Work["Quarter"] = dfSalesDataML_Work["Date"].dt.quarter

#Thanks to StackOverflow user chimpsarehungry for the solution
dfSalesDataML_Work["MonthSin"] = np.sin(2 * np.pi * dfSalesDataML_Work["Month"] / 12)
dfSalesDataML_Work["MonthCos"] = np.cos(2 * np.pi * dfSalesDataML_Work["Month"] / 12)

dfSalesDataML_Work["IsHoliday"] = dfSalesDataML_Work["IsHoliday"].astype(int)

#load linear regression pipeline
objPipeline = joblib.load(CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_TEST_PATH)

#set year range
intYear= 2012
intPreviousYear = intYear - 1

dfTest = dfSalesDataML_Work.copy()
dfTest = dfTest[dfTest["Year"] == intYear]

dfPrevious = dfSalesDataML_Work.copy()
dfPrevious = dfPrevious[dfPrevious["Year"] == intPreviousYear]

#show year range
print(f"Test Year: {intYear}")
print(f"Previous Year: {intPreviousYear}")
print()

#show row data
print(f"Test Rows: {len(dfTest)}")
print(f"Previous Year Rows: {len(dfPrevious)}")
print()

#configure features
lstFeatures = [
    "Store",
    "Dept",
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

#create model prediction
dfXTest = dfTest[lstFeatures]
dfTest["Predicted_Sales"] = objPipeline.predict(dfXTest)


#get actual sales to compare
dfActualSales = (
    dfTest
    .groupby(
        ["Date", "WeekOfYear"]
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure we don't get a clash fof names in the plot!
dfActualSales = dfActualSales.rename(
    columns={
        "Weekly_Sales": "Actual_Sales"
    }
)


#get predicted sales to compare
dfPredictedSales = (
    dfTest
    .groupby(
        ["Date", "WeekOfYear"]
    )["Predicted_Sales"]
    .sum()
    .reset_index()
)


#get previous sales by week to compare
dfPreviousSales = (
    dfPrevious
    .groupby(
        "WeekOfYear"
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure we don't get a clash of names in the plot!
dfPreviousSales = dfPreviousSales.rename(
    columns={
        "Weekly_Sales": "Previous_Year_Sales"
    }
)


#merge plots
dfPlot = dfActualSales.merge(
    dfPredictedSales,
    on=["Date", "WeekOfYear"],
    how="left"
)

#merge previous year
dfPlot = dfPlot.merge(
    dfPreviousSales,
    on="WeekOfYear",
    how="left"
)


#sort by Date
dfPlot = dfPlot.sort_values("Date")

#show data
print("Plot Data First 10 Rows:")

print(
    dfPlot[
        [
            "Date",
            "WeekOfYear",
            "Actual_Sales",
            "Predicted_Sales",
            "Previous_Year_Sales"
        ]
    ].head(20)
)
print()

#check for missing previous year values
dfMissingPrevious = dfPlot["Previous_Year_Sales"].isna().sum()

print(f"Number of Missing Previous Year Values: {dfMissingPrevious}")
print()

#create plot
plt.figure(figsize=(15, 7))
plt.plot(
    dfPlot["Date"],
    dfPlot["Actual_Sales"],
    label="Actual Sales",
    color="black",
    linewidth=2
)

#plot predictions
plt.plot(
    dfPlot["Date"],
    dfPlot["Predicted_Sales"],
    label="Linear Regression Prediction",
    color="red",
    linewidth=2
)


#plot previous year sales as comparison
plt.plot(
    dfPlot["Date"],
    dfPlot["Previous_Year_Sales"],
    label="Previous Year Sales",
    color="blue",
    linestyle="--",
    linewidth=2
)


#configure plot
plt.title("Sales - Linear Regression vs Previous Year", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Weekly Sales", fontsize=12)
#test using best fit
plt.legend(loc="best") 
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()


#save plot image to report folder
plt.savefig(
    CNST_STR_LINEAR_PLOT_HYPOTHESIS12_TEST_PATH,
    dpi=300,
    bbox_inches="tight"
)

print(f"Plot Saved To: {CNST_STR_LINEAR_PLOT_HYPOTHESIS12_TEST_PATH}")

#show plot
plt.show()

# Random Forest - Hypothesis 12 - Test

First run a test comparing predicitons for last years values with the actual values to see how accurate the model is

In [21]:
#random forest test compare last years sales with predicted values for same year
#saves pipeline into main pipelines folder and the same in the streamlit folder

dfSalesDataML_Work = dfSalesDataML_Work.dropna(subset=["Date"])

#sprinkle some feature engineering
dfSalesDataML_Work["Year"] = dfSalesDataML_Work["Date"].dt.year
dfSalesDataML_Work["Month"] = dfSalesDataML_Work["Date"].dt.month
dfSalesDataML_Work["Day"] = dfSalesDataML_Work["Date"].dt.day
dfSalesDataML_Work["DayOfWeek"] = dfSalesDataML_Work            ["Date"].dt.dayofweek
dfSalesDataML_Work["WeekOfYear"] = dfSalesDataML_Work["Date"].dt.isocalendar().week.astype(int)
dfSalesDataML_Work["Quarter"] = dfSalesDataML_Work["Date"].dt.quarter

#cyclical month features thanks to StackOverflow user chimpsarehungry for the solution
dfSalesDataML_Work["MonthSin"] = np.sin(2 * np.pi * dfSalesDataML_Work["Month"] / 12)
dfSalesDataML_Work["MonthCos"] = np.cos(2 * np.pi * dfSalesDataML_Work["Month"] / 12)
dfSalesDataML_Work["IsHoliday"] = dfSalesDataML_Work["IsHoliday"].astype(int)


#create and train model
#training:
#ll years before 2012
#
#Testing:
#2012 only
#
#2013 is deliberately NOT used for testing.

dfTrain = dfSalesDataML_Work.copy()
dfTrain = dfTrain[dfTrain["Year"] < 2012]
dfTest = dfSalesDataML_Work.copy()
dfTest = dfTest[dfTest["Year"] == 2012]

#show row data
print(f"Training Rows: {len(dfTrain)}")
print(f"Testing Rows: {len(dfTest)}")
print()

#show date range
print(f"Training Date Range: {dfTrain['Date'].min()} To {dfTrain['Date'].max()}")
print(f"Testing Date Range: {dfTest['Date'].min()} To {dfTest['Date'].max()}")
print()

#configure features and target
lstFeatures = [
    "Store",
    "Dept",
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

#set target
strTarget = "Weekly_Sales"

#setup X and y for training and testing
dfXTrain = dfTrain[lstFeatures]
dfyTrain = dfTrain[strTarget]

dfXTest = dfTest[lstFeatures]
dfyTest = dfTest[strTarget]


#set numeric features
lstNumericFeatures = [
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

#set categorical features
lstCategoricalFeatures = [
    "Store",
    "Dept"
]

#set numeric transformer
objNumericTransformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

#set categorical transformer
objCategoricalTransformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

#set column transformer
objPreProcessor= ColumnTransformer(
    transformers=[
        (
            "numeric",
            objNumericTransformer,
            lstNumericFeatures
        ),
        (
            "categorical",
            objCategoricalTransformer,
            lstCategoricalFeatures
        )
    ]
)

#create random forest model
objModel = RandomForestRegressor(
    n_estimators=20,     #50
    max_depth=20,      #None
    min_samples_split=2, #2
    min_samples_leaf=1,  #1
    random_state=42,     
    n_jobs=-1
)

#create pipeline
objPipeline = Pipeline(
    steps=[
        ("preprocessor", objPreProcessor),
        ("model", objModel)
    ]
)

#train model
print("Training Random Forest...")

objPipeline.fit(
    dfXTrain,
    dfyTrain
)
print()

#get prediction==================================================
objyPred = objPipeline.predict(dfXTest)


#evaluate results
fltMAE = mean_absolute_error(dfyTest, objyPred)
fltRMSE = np.sqrt(mean_squared_error(dfyTest, objyPred))
fltR2 = r2_score(dfyTest, objyPred)

#print results
print("Random Forest Results")
print("=" * 30)

print(f"Mean Absolute Error (MAE):  {fltMAE:,.2f}")
print(f"Root Mean Squared Error (RMSE): {fltRMSE:,.2f}")
print(f"R-Squared (R²):   {fltR2:.4f}")
print()

#show OLS model summary for Weekly_Sales
ols_model = smf.ols("Weekly_Sales ~ Store + Dept + IsHoliday + Year + Month + Day + DayOfWeek + WeekOfYear + Quarter + MonthSin + MonthCos", 
                    data=dfSalesDataML_Work).fit()
print(ols_model.summary())
print()

#save piepline to main folder
joblib.dump(
    objPipeline,
    CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_PATH
)

#save to streamlit folder
joblib.dump(
    objPipeline,
    CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH
)

print("Pipeline Saved To:")
print(CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_PATH)
print(CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_STREAMLIT_PATH)

Training Rows: 294132
Testing Rows: 127438

Training Date Range: 2010-02-05 00:00:00 To 2011-12-30 00:00:00
Testing Date Range: 2012-01-06 00:00:00 To 2012-10-26 00:00:00

Training Random Forest...

Random Forest Results
Mean Absolute Error (MAE):  7,424.41
Root Mean Squared Error (RMSE): 10,070.20
R-Squared (R²):   0.7928

                            OLS Regression Results                            
Dep. Variable:           Weekly_Sales   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     1386.
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:47:19   Log-Likelihood:            -4.8200e+06
No. Observations:              421570   AIC:                         9.640e+06
Df Residuals:                  421559   BIC:                         9.640e+06
Df Model:                          10     

# Observations - Random Forest Model - Hypothesis 12 - Test

The R Squared value is 0.7928 which is far enough from 0 to suggest the model is a good fit for the data  

The Mean Absolute Error (MAE) is 7,424.41 which means on average the model predicted sales differed from the actual sales by  
approximately 7,424.41  per store and department, considering the size of average stores sales not too bad a variance,  
however it is lower than the linear regression model 

Root Mean Squared Error (RMSE) is 10,070.20 the fact it is higher than MAE suggests that while most predications are  
close to the actual values, there are some occasion where the model is less accurate, this value is lower than the value for  
linear regression model, more evidence this is the model to use  

These are usually caused by outliers

The OLS regression summary shows that a model based soley on Weekly_Sales shows no significant correlation with the other   
variables in the model, which is to be expected as the model is based on Weekly_Sales, but it shown here for completeness

# Random Forest - Plot - Hypothesis 12 - Test

Visualise the model results


In [ ]:
#random forest plot test compare last years sales with predicted values for same year

dfSalesDataML_Work = dfSalesDataML_Work.dropna(subset=["Date"])

#sprinkle some feature engineering
dfSalesDataML_Work["Year"] = dfSalesDataML_Work["Date"].dt.year
dfSalesDataML_Work["Month"] = dfSalesDataML_Work["Date"].dt.month
dfSalesDataML_Work["Day"] = dfSalesDataML_Work["Date"].dt.day
dfSalesDataML_Work["DayOfWeek"] = dfSalesDataML_Work["Date"].dt.dayofweek
dfSalesDataML_Work["WeekOfYear"] = dfSalesDataML_Work["Date"].dt.isocalendar().week.astype(int)
dfSalesDataML_Work["Quarter"] = dfSalesDataML_Work["Date"].dt.quarter

#cyclical month features thanks to StackOverflow user chimpsarehungry for the solution
dfSalesDataML_Work["MonthSin"] = np.sin(2 * np.pi * dfSalesDataML_Work["Month"] / 12)
dfSalesDataML_Work["MonthCos"] = np.cos(2 * np.pi * dfSalesDataML_Work["Month"] / 12)

dfSalesDataML_Work["IsHoliday"] = dfSalesDataML_Work["IsHoliday"].astype(int)

#load pipeline for random forest
objPipeline = joblib.load(CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_TEST_PATH)

#set year range
intYear= 2012
intPreviousYear = intYear - 1
dfTest = dfSalesDataML_Work.copy()
dfTest = dfTest[dfTest["Year"] == intYear]

dfPrevious = dfSalesDataML_Work.copy()
dfPrevious = dfPrevious[dfPrevious["Year"] == intPreviousYear].copy()

#show year range
print(f"Test Year: {intYear}")
print(f"Previous Year: {intPreviousYear}")
print()

#show rows
print(f"Test Rows: {len(dfTest)}")
print(f"Previous Year Rows: {len(dfPrevious)}")
print()

#configure features
lstFeatures = [
    "Store",
    "Dept",
    "IsHoliday",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos"
]

#get predictions
dfXTest = dfTest[lstFeatures]
dfTest["Predicted_Sales"] = objPipeline.predict(dfXTest)

#get actual sales
dfActualSales = (
    dfTest
    .groupby(
        ["Date", "WeekOfYear"]
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure no name clashes in the plot!
dfActualSales = dfActualSales.rename(
    columns={
        "Weekly_Sales": "Actual_Sales"
    }
)

#get predicted sales
dfPredictedSales = (
    dfTest
    .groupby(
        ["Date", "WeekOfYear"]
    )["Predicted_Sales"]
    .sum()
    .reset_index()
)

#get previous year sales
dfPreviousSales = (
    dfPrevious
    .groupby(
        "WeekOfYear"
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure no name clashes in the plot!
dfPreviousSales = dfPreviousSales.rename(
    columns={
        "Weekly_Sales": "Previous_Year_Sales"
    }
)

#merge DataFrames
dfPlot = dfActualSales.merge(
    dfPredictedSales,
    on=["Date", "WeekOfYear"],
    how="left"
)

#merge DataFrames
dfPlot = dfPlot.merge(
    dfPreviousSales,
    on="WeekOfYear",
    how="left"
)


#Sort by date
dfPlot = dfPlot.sort_values("Date")

#show data
print("Plot Data First 10 Rows:")

print(
    dfPlot[
        [
            "Date",
            "WeekOfYear",
            "Actual_Sales",
            "Predicted_Sales",
            "Previous_Year_Sales"
        ]
    ].head(20)
)
print()


#check missing previous-year values
dfMissingPrevious = (
    dfPlot["Previous_Year_Sales"]
    .isna()
    .sum()
)

print(f"Missing Previous Year Values: {dfMissingPrevious}")
print()


#create plot
plt.figure(figsize=(15, 7))

#create actual sales plot
plt.plot(
    dfPlot["Date"],
    dfPlot["Actual_Sales"],
    label="Actual Sales",
    color="black",
    linewidth=2
)

#plot predictions
plt.plot(
    dfPlot["Date"],
    dfPlot["Predicted_Sales"],
    label="Random Forest Prediction",
    color="red",
    linewidth=2
)

#plot previous years sales
plt.plot(   
    dfPlot["Date"],
    dfPlot["Previous_Year_Sales"],
    label="Previous Year Sales",
    color="blue",
    linestyle="--",
    linewidth=2
)

#format plot
plt.title("Sales - Random Forest vs Previous Year", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Weekly Sales", fontsize=12)
#prefer best fit
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
#make sure ticks are readable
plt.xticks(rotation=45)
plt.tight_layout()

#save plot
plt.savefig(
    CNST_STR_FOREST_PLOT_HYPOTHESIS12_TEST_PATH,
    dpi=300,
    bbox_inches="tight"
)

#show plot
plt.show()

# Linear Regression Model - Hypothesis 12 - Plot

Predict 2013 Sales


In [ ]:
#linear regression plot predict next years sales 
#saves pipeline into main pipelines folder and the same in the streamlit folder



#feature engineering function
def funcCreateFeatures(df):

    df = df.copy()
    #date features
    df["Year"] = ( df["Date"].dt.year)
    df["Month"] = ( df["Date"].dt.month)
    df["Day"] = (df["Date"].dt.day)
    df["DayOfWeek"] = (df["Date"].dt.dayofweek)
    df["WeekOfYear"] = (df["Date"]
        .dt.isocalendar()
        .week
        .astype(int)
    )
    df["Quarter"] = ( df["Date"].dt.quarter)
    #cyclical features thanks to StackOverflow user chimpsarehungry for the solution
    df["MonthSin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["MonthCos"] = np.cos(2 * np.pi * df["Month"] / 12)
    df["WeekSin"] = np.sin(2 * np.pi * df["WeekOfYear"] / 52)
    df["WeekCos"] = np.cos(2 * np.pi * df["WeekOfYear"] / 52)

    #holiday
    df["IsHoliday"] = (df["IsHoliday"].astype(int))

    return df



#load csv files
dfFeatures = dictDataFrames["Features_DataSet_Visualisation.csv"]
dfSales = dictDataFrames["Sales_DataSet_Visualisation.csv"]
dfStores = dictDataFrames["Stores_DataSet_Visualisation.csv"]
#convert Date to datetime
dfFeatures["Date"] = pd.to_datetime(
    dfFeatures["Date"],
    errors="coerce"
)

dfSales["Date"] = pd.to_datetime(
    dfSales["Date"],
    errors="coerce"
)


#remove invalid dates
dfFeatures = dfFeatures.dropna(
    subset=["Date"]
)

dfSales = dfSales.dropna(
    subset=["Date"]
)

#show date rnage
print("Features Date Range:")
print(f"{dfFeatures['Date'].min()} to {dfFeatures['Date'].max()}")
print()

print("Sales Date Range:")
print(f"{dfSales['Date'].min()} to {dfSales['Date'].max()}")
print()

#merge features and stores
dfFeaturesStores = dfFeatures.merge(
    dfStores,
    on="Store",
    how="left"
)

#show new shape
print(f"Features + Stores Shape: {dfFeaturesStores.shape}")
print()
#create training data

#merge sales with features
dfTrain = dfSales.merge(
    dfFeaturesStores,
    on=[
        "Store",
        "Date",
        "IsHoliday"
    ],
    how="left"
)

#show shape
print(f"Training Dataset Shape: {dfTrain.shape}")
print()

#create forecast datas
dfForecast = dfFeaturesStores[
    dfFeaturesStores["Date"] >
    dfSales["Date"].max()
].copy()

#we need predictions for every Store/Department!
#get rid of duplicates
dfStoreDepts = dfSales[ ["Store", "Dept"] ].drop_duplicates()

#merge forecast with departments 
dfForecast = dfForecast.merge(
    dfStoreDepts,
    on="Store",
    how="inner"
)

#show shape
print(f"Forecast Dataset: {dfForecast.shape}")
print()

#show date range
print("Forecast Date Range:")
print(f"{dfForecast['Date'].min()} to {dfForecast['Date'].max()}")
print()

#configure DataFrames
dfTrain= funcCreateFeatures(dfTrain)
dfForecast = funcCreateFeatures(dfForecast)

#define markdown columns
lstMarkdown = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]


for column in lstMarkdown:
    if column in dfTrain.columns:
       dfTrain[column] = (
            dfTrain[column]
            .fillna(0)
       )

    if column in dfForecast.columns:
       dfForecast[column] = (
            dfForecast[column]
            .fillna(0)
       )


#define features
lstFeatures = [

    # Store information
    "Store",
    "Dept",
    "Store_Type",
    "Size",

    # Date information
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",

    # Cyclical features
    "MonthSin",
    "MonthCos",
    "WeekSin",
    "WeekCos",

    # Holiday
    "IsHoliday",

    # Walmart economic features
    "Temperature",
    "Unemployment",

    # Markdown
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

#define target
strTarget = "Weekly_Sales"

#check columns
objMissingColumns = [
    column
    for column in lstFeatures
    if column not in dfTrain.columns
]

#missing columns?
if objMissingColumns:
    raise ValueError(
        f"Missing columns: {objMissingColumns}"
    )


#setup X and y for training and forecasting
dfXTrain = dfTrain[lstFeatures]
dfyTrain = dfTrain[strTarget]
dfXForecast = dfForecast[lstFeatures]

#define numeric features
lstNumericFeatures = [
    "Size",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos",
    "WeekSin",
    "WeekCos",
    "IsHoliday",
    "Temperature",
    "Unemployment",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

#define categorical features
lstCategoricalFeatures = [
    "Store",
    "Dept",
    "Store_Type"
]

#configure numeric transformer
objNumericTransformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)

#configure categorical transformer
objCategoricalTransformer = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

#configure column transformer
objPreProcessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            objNumericTransformer,
            lstNumericFeatures
        ),

        (
            "categorical",
            objCategoricalTransformer,
            lstCategoricalFeatures
        )
    ]
)

#create linear regression model
objModel = LinearRegression()

#configure pipeline
objPipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            objPreProcessor
        ),
        (
            "model",
            objModel
        )
    ]
)


#train model
print("Training Linear Regression...")
objPipeline.fit(dfXTrain, dfyTrain)

print("Training complete")
print()

#save pipeline to main folder
joblib.dump(
    objPipeline,
    CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_PATH
)

#save to sreamlit folder
joblib.dump(
    objPipeline,
    CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH
)

print("Pipeline Saved To:")

print(CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_PATH)
print(CNST_STR_LINEAR_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH)
print()

#start prediction!
print("Generating 2013 Sales Predictions:")
print()

dfForecast["Predicted_Sales"] = (
    objPipeline.predict(
        dfXForecast
    )
)


#select 2013 predictions
dfTest = dfForecast.copy()
dfTest = dfTest[ dfTest["Date"].dt.year == 2013]

#show rows
print(f"2013 Sales Prediction Rows: {len(dfTest)}")
print()

#show date range
print("2013 Sales Prediction Period:")
print(f"{dfTest['Date'].min()} to {dfTest['Date'].max()}")
print()

#aggregate prediction values
dfForecast = (
    dfTest
    .groupby(
        ["Date", "WeekOfYear"]
    )["Predicted_Sales"]
    .sum()
    .reset_index()
)


#get actual sales for 2012 to compare with 2013 predictions
dfActualSales = dfTrain.copy()
dfActualSales = dfActualSales[
    dfActualSales["Year"] == 2012
]

#aggregate actual sales for 2012
dfPrevious = (
    dfActualSales
    .groupby(
        "WeekOfYear"
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure we don't get a clash of names in the plot!
dfPrevious = dfPrevious.rename(
    columns={
        "Weekly_Sales":
        "Previous_Year_Sales"
    }
)


#merge forecast with previous year sales for comparison
dfPlot= dfForecast.merge(
    dfPrevious,
    on="WeekOfYear",
    how="left"
)

#sort by Date
dfPlot = dfPlot.sort_values("Date")

#show OLS summary for predicted sales vs previous year sales
ols_model = smf.ols("Predicted_Sales ~ WeekOfYear + Previous_Year_Sales", 
                    data=dfPlot).fit()
print(ols_model.summary())
print()


#create plot
plt.figure(figsize=(15, 7))

plt.plot(
    dfPlot["Date"],
    dfPlot["Predicted_Sales"],
    color="red",
    linewidth=2,
    label="2013 Predicted Sales"
)

plt.title(
    "Linear Regression - 2013 Sales Forecast",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.legend()
plt.grid(alpha=0.3)
#make sure ticks are readable
plt.xticks(rotation=45)
plt.tight_layout()

#save plot to report folder
plt.savefig(
    CNST_STR_LINEAR_PLOT_HYPOTHESIS12_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("Plot Saved!")
print(CNST_STR_LINEAR_PLOT_HYPOTHESIS12_PATH)
print()

plt.show()

# Random Forest Model - Hypothesis 12 - Plot

In [ ]:
#random forest predict next years sales
#saves pipeline into main pipelines folder and the same in the streamlit folder


def funcCreateFeatures(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfWeek"] = (
        df["Date"].dt.dayofweek
    )

    df["WeekOfYear"] = (
        df["Date"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    df["Quarter"] = (
        df["Date"].dt.quarter
    )

    df["MonthSin"] = np.sin(
        2 * np.pi * df["Month"] / 12
    )

    df["MonthCos"] = np.cos(
        2 * np.pi * df["Month"] / 12
    )

    df["WeekSin"] = np.sin(
        2 * np.pi *
        df["WeekOfYear"] / 52
    )

    df["WeekCos"] = np.cos(
        2 * np.pi *
        df["WeekOfYear"] / 52
    )

    df["IsHoliday"] = (
        df["IsHoliday"]
        .astype(int)
    )

    return df


#load csv files
dfFeatures = dictDataFrames["Features_DataSet_Visualisation.csv"]
dfSales = dictDataFrames["Sales_DataSet_Visualisation.csv"]
dfStores = dictDataFrames["Stores_DataSet_Visualisation.csv"]

#convert Date to datetime
dfFeatures["Date"] = pd.to_datetime(
    dfFeatures["Date"],
    errors="coerce"
)

dfSales["Date"] = pd.to_datetime(
    dfSales["Date"],
    errors="coerce"
)

#delete rows with invalid dates
dfFeatures = dfFeatures.dropna(
    subset=["Date"]
)

dfSales = dfSales.dropna(
    subset=["Date"]
)

#show date range
print(f"Sales: {dfSales['Date'].min()} to {dfSales['Date'].max()}")
print()

print(
    f"Features: {dfFeatures['Date'].min()} to {dfFeatures['Date'].max()}"
)
print()

#merge features and stores
dfFeaturesStores = dfFeatures.merge(
    dfStores,
    on="Store",
    how="left"
)

#,erge sales and featuresstores
dfTraining = dfSales.merge(
    dfFeaturesStores,
    on=[
        "Store",
        "Date",
        "IsHoliday"
    ],
    how="left"
)


#create forecast data
dfForecast = dfFeaturesStores.copy()
dfForecast[
    dfForecast["Date"] >
    dfSales["Date"].max()
]

dfStoreDepts = dfSales[
    ["Store", "Dept"]
].drop_duplicates()


dfForecast = dfForecast.merge(
    dfStoreDepts,
    on="Store",
    how="inner"
)

#show rows
print(f"Training Rows: {len(dfTraining)}")
print(f"Forecast Rows: {len(dfForecast)}")
print()

#sprinkle some feature engineering
dfTrain = funcCreateFeatures(dfTrain)
dfForecast = funcCreateFeatures(dfForecast)

#configure markdown columns
lstMarkdown = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

#process
for column in lstMarkdown:
    if column in dfTraining.columns:
        dfTraining[column] = (
            dfTraining[column]
            .fillna(0)
        )

    if column in dfForecast.columns:
        dfForecast[column] = (
            dfForecast[column]
            .fillna(0)
        )


#configure features
lstFeatures = [
    "Store",
    "Dept",
    "Store_Type",
    "Size",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos",
    "WeekSin",
    "WeekCos",
    "IsHoliday",
    "Temperature",
    "Unemployment",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

#setup train
dfXtrain = dfTrain[
    lstFeatures
]

dfyTrain = dfTrain[
    "Weekly_Sales"
]

dfXForecast = dfForecast[
    lstFeatures
]


#configure numeric features
lstNumericFeatures = [
    "Size",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "MonthSin",
    "MonthCos",
    "WeekSin",
    "WeekCos",
    "IsHoliday",
    "Temperature",
    "Unemployment",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

#configure categorical features
lstCategoricalFeatures = [
    "Store",
    "Dept",
    "Store_Type"
]

#configure numeric transformer
objNumericTransformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

#configure categorical transformer
objCategoricalTransformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

#configure column transformer
objPreProcessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            objNumericTransformer,
            lstNumericFeatures
        ),

        (
            "categorical",
            objCategoricalTransformer,
            lstCategoricalFeatures
        )
    ]
)


#create model
objModel = RandomForestRegressor(
    n_estimators=20,  #50
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

#create pipeline
objPipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            objPreProcessor
        ),

        (
            "model",
            objModel
        )
    ]
)


#train model
print("Training Random Forest...")

objPipeline.fit(
    dfXtrain,
    dfyTrain
)

print("Training Complete")
print()

#save pipeline to main folder
joblib.dump(
    objPipeline,
    CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_PATH
)

#save to streamlit folder
joblib.dump(
    objPipeline,
    CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH
)

print("Pipeline Saved To:")
print(CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_PATH)
print(CNST_STR_FOREST_PIPELINE_HYPOTHESIS12_STREAMLIT_PATH)
print()

#run prediction for 2013 sales
print("Generating Predictions...")
print()
dfForecast["Predicted_Sales"] = (
    objPipeline.predict(
        dfXForecast
    )
)

#get 2013 data
dfPredictedSales = dfForecast.copy()
dfPredictedSales = dfPredictedSales[dfPredictedSales["Date"].dt.year == 2013]

#print data rows count
print(f"Amount of 2013 Rows: {len(dfPredictedSales)}")
print()

print("2013 Period:")
print(f"{dfPredictedSales['Date'].min()} to {dfPredictedSales['Date'].max()}")
print()

#aggreagate sales
dfForecastSales = (
    dfPredictedSales
    .groupby(
        ["Date", "WeekOfYear"]
    )["Predicted_Sales"]
    .sum()
    .reset_index()
)

#get 2012 sales
dfActualSales = dfTrain.copy()
dfActualSales= dfActualSales[dfActualSales["Year"] == 2012]

#get previous years sales
dfPreviousSales = (
    dfActualSales
    .groupby(
        "WeekOfYear"
    )["Weekly_Sales"]
    .sum()
    .reset_index()
)

#make sure we don't get a clash of names in the plot!
dfPreviousSales = dfPreviousSales.rename(
    columns={
        "Weekly_Sales":
        "Previous_Year_Sales"
    }
)

#merge forecastsales with previoussales
dfPlot = dfForecastSales.merge(
    dfPreviousSales,
    on="WeekOfYear",
    how="left"
)

#sort by Date
dfPlot = dfPlot.sort_values("Date")

#show OLS summary for predicted sales vs previous year sales
ols_model = smf.ols("Predicted_Sales ~ WeekOfYear + Previous_Year_Sales", 
                    data=dfPlot).fit()
print(ols_model.summary())
print()

#create plot
plt.figure(figsize=(15, 7))

plt.plot(
    dfPlot["Date"],
    dfPlot["Predicted_Sales"],
    color="red",
    linewidth=2,
    label="2013 Predicted Sales"
)

plt.title(
    "Random Forest - 2013 Sales Forecast",
    fontsize=16
)

plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.legend()

plt.grid(alpha=0.3)
#make sure ticks are readable
plt.xticks(rotation=45)
plt.tight_layout()

#save image to report folder
plt.savefig(
    CNST_STR_FOREST_PLOT_HYPOTHESIS12_PATH,
    dpi=300,
    bbox_inches="tight"
)


print("Plot Saved To:")
print(CNST_STR_FOREST_PLOT_HYPOTHESIS12_PATH)
print()

plt.show()

# Observations - Regression Types Hypothesis 12 - Plot

While the models are close in performance, the Random Forest model demonstrates a slight edge in predictive accuracy for sales forecasting  
with an R squared

# Conclusions - Hypothesis 12

While the models are close in performance, the Random Forest model demonstrates a slight edge in predictive accuracy for sales forecasting  
The plots looks closer to the mean than the linear regression model, and I prefer the closeness of the random forest's R Squared value

Running the OLS summary for Weekly_Sales for both linear regression and random forest was interesting as even though technically the results 
are meaningless they were very close to each other and not so close to zero that Hypothesis Zero would be true!